# Lab 5 — Polynomial Regression
**Linear Algebra · UATX**

Given $N$ data points $(x_i, y_i)$, fitting a degree-$d$ polynomial is solving the overdetermined system $X\beta \approx y$ where $X$ is the **Vandermonde matrix** $X_{ij} = x_i^j$. The **normal equations** $X^\top X \beta = X^\top y$ give the least-squares solution.

**Tasks**
1. Build the Vandermonde matrix and fit degree 2; recover true coefficients.
2. Compare degrees $d = 1, 3, 5, 10$; observe underfitting and overfitting.
3. Fit degree 11 with only 12 data points; compute rank and kernel dimension.
4. Use a validation set to find the optimal degree.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

## §1  Setup: Vandermonde Matrix and Normal Equations

In [ ]:
def generate_data(N, poly_fn, x_min=-5, x_max=5, noise=0.5, seed=None):
    """Generate N noisy samples from poly_fn."""
    if seed is not None: np.random.seed(seed)
    x = np.random.uniform(x_min, x_max, N)
    y = poly_fn(x) + noise * np.random.randn(N)
    return x, y

def feature_matrix(x, d):
    """Vandermonde matrix: X[i,j] = x[i]^j,  shape (N, d+1)."""
    return np.column_stack([x**j for j in range(d+1)])

def fit_polynomial(x, y, d):
    """Solve normal equations X^T X beta = X^T y."""
    X = feature_matrix(x, d)
    beta = np.linalg.lstsq(X, y, rcond=None)[0]   # numerically stable
    return beta, X

def poly_predict(beta, x_new):
    d = len(beta) - 1
    X_new = feature_matrix(x_new, d)
    return X_new @ beta


# True polynomial: p(x) = 1 + 2x + x^2
true_poly = lambda x: 1 + 2*x + x**2
N = 100
x_train, y_train = generate_data(N, true_poly, seed=42)

# Fit degree 2
beta2, X2 = fit_polynomial(x_train, y_train, d=2)
print(f'True coefficients:    [1, 2, 1]')
print(f'Recovered (degree 2): {np.round(beta2, 4)}')

x_plot = np.linspace(-5, 5, 300)
plt.figure(figsize=(7, 4))
plt.scatter(x_train, y_train, s=10, alpha=0.4, label='data')
plt.plot(x_plot, poly_predict(beta2, x_plot), 'r-', lw=2, label='fit (d=2)')
plt.xlabel('x'); plt.ylabel('y'); plt.legend(); plt.grid(True)
plt.title('Degree-2 fit on noisy data from $p(x)=1+2x+x^2$')
plt.tight_layout(); plt.show()

## §2  Effect of Degree: Underfitting and Overfitting

In [ ]:
degrees = [1, 2, 3, 5, 10]
fig, axes = plt.subplots(1, len(degrees), figsize=(18, 3.5))

for ax, d in zip(axes, degrees):
    beta_d, X_d = fit_polynomial(x_train, y_train, d)
    y_fit = poly_predict(beta_d, x_train)
    train_err = np.sqrt(np.mean((y_fit - y_train)**2))   # RMSE

    ax.scatter(x_train, y_train, s=5, alpha=0.3)
    ax.plot(x_plot, poly_predict(beta_d, x_plot), 'r-', lw=2)
    ax.set_title(f'd={d}\nRMSE={train_err:.3f}')
    ax.set_ylim(-5, 35); ax.grid(True)
    ax.set_xlabel('x')

plt.suptitle('Polynomial fits of increasing degree')
plt.tight_layout(); plt.show()

In [ ]:
# Training RMSE vs degree
degrees_all = list(range(1, 15))
train_rmse = []
for d in degrees_all:
    beta_d, X_d = fit_polynomial(x_train, y_train, d)
    y_fit = poly_predict(beta_d, x_train)
    train_rmse.append(np.sqrt(np.mean((y_fit - y_train)**2)))

plt.figure(figsize=(7, 4))
plt.plot(degrees_all, train_rmse, 'b-o', lw=2, label='Training RMSE')
plt.axhline(0.5, color='gray', ls='--', label='noise level $\\sigma=0.5$')
plt.xlabel('Degree $d$'); plt.ylabel('RMSE')
plt.title('Training error keeps decreasing — that does not mean the fit improves!')
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

## §3  Overfitting: $N=12$ Points, Degree 11

When $d \geq N-1$ the Vandermonde matrix is square; the fit interpolates every point exactly — this is overfitting.

In [ ]:
N_small = 12
d_overfit = 11
x_sm, y_sm = generate_data(N_small, true_poly, seed=0)

beta_of, X_of = fit_polynomial(x_sm, y_sm, d_overfit)
print(f'X shape: {X_of.shape}')
print(f'rank(X): {np.linalg.matrix_rank(X_of)}')
print(f'dim ker(X): {X_of.shape[1] - np.linalg.matrix_rank(X_of)}')
print(f'Training RMSE: {np.sqrt(np.mean((X_of @ beta_of - y_sm)**2)):.2e}')

plt.figure(figsize=(8, 4))
plt.scatter(x_sm, y_sm, s=60, zorder=5, label='12 data points')
plt.plot(x_plot, poly_predict(beta_of, x_plot), 'r-', lw=2, label=f'd={d_overfit} (interpolates!)')
beta_true, _ = fit_polynomial(x_sm, y_sm, 2)
plt.plot(x_plot, poly_predict(beta_true, x_plot), 'g--', lw=2, label='d=2 (correct)')
plt.ylim(-10, 40); plt.xlabel('x'); plt.ylabel('y'); plt.legend(); plt.grid(True)
plt.title('Overfitting: degree 11 interpolates 12 points exactly')
plt.tight_layout(); plt.show()

## §4  Validation Set: Finding the Right Degree

In [ ]:
# Training and validation sets from same distribution
N_train, N_val = 50, 50
np.random.seed(1)
x_tr, y_tr   = generate_data(N_train, true_poly, seed=1)
x_val, y_val = generate_data(N_val,   true_poly, seed=2)

degrees_all = list(range(1, 16))
train_rmse, val_rmse = [], []

for d in degrees_all:
    beta_d, _ = fit_polynomial(x_tr, y_tr, d)
    y_tr_pred  = poly_predict(beta_d, x_tr)
    y_val_pred = poly_predict(beta_d, x_val)
    train_rmse.append(np.sqrt(np.mean((y_tr_pred  - y_tr )**2)))
    val_rmse.append(  np.sqrt(np.mean((y_val_pred - y_val)**2)))

best_d = degrees_all[np.argmin(val_rmse)]
print(f'Best degree by validation RMSE: d = {best_d}')

plt.figure(figsize=(7, 4))
plt.plot(degrees_all, train_rmse, 'b-o', lw=2, label='Training RMSE')
plt.plot(degrees_all, val_rmse,   'r-s', lw=2, label='Validation RMSE')
plt.axvline(best_d, color='green', ls='--', lw=2, label=f'Best d={best_d}')
plt.axhline(0.5, color='gray', ls=':', label='$\\sigma=0.5$')
plt.xlabel('Degree $d$'); plt.ylabel('RMSE')
plt.title('Training vs Validation error — the bias–variance tradeoff')
plt.legend(); plt.grid(True); plt.ylim(0, 3)
plt.tight_layout(); plt.show()

In [ ]:
# Why does validation error eventually explode?
# Because the Vandermonde matrix becomes ill-conditioned.
cond_numbers = []
for d in degrees_all:
    X_d = feature_matrix(x_tr, d)
    cond_numbers.append(np.linalg.cond(X_d))

plt.figure(figsize=(7, 3))
plt.semilogy(degrees_all, cond_numbers, 'k-o', lw=2)
plt.axhline(1/np.finfo(float).eps, color='red', ls='--', label='$1/\\epsilon_{\\rm mach}$')
plt.xlabel('Degree $d$'); plt.ylabel('cond($X$)')
plt.title('Condition number of the Vandermonde matrix')
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()